In [0]:
# --------------------------------------------------
# 1. GET CURRENT RUN
# --------------------------------------------------

running_runs = spark.sql("""
    SELECT
        run_id,
        batch_id
    FROM workspace.control.etl_run_log
    WHERE pipeline_name = 'orders_pipeline'
      AND status = 'RUNNING'
""").collect()

if len(running_runs) != 1:
    raise ValueError(
        f"Expected exactly 1 RUNNING run, found {len(running_runs)}"
    )

run_id = running_runs[0]["run_id"]
batch_id = running_runs[0]["batch_id"]

print(f"Run ID:   {run_id}")
print(f"Batch ID: {batch_id}")


# --------------------------------------------------
# 2. PROCESS BRONZE
# --------------------------------------------------

try:

    # Count before load
    bronze_before = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.bronze.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]


    spark.sql(f"""
    INSERT INTO workspace.bronze.orders

    SELECT
        l.order_id,
        l.customer_id,
        l.amount,
        l.status,
        l.order_date,
        l.last_updated,
        l.event_id,
        l.batch_id,
        l.source_file,
        l.load_timestamp

    FROM workspace.landing.orders l

    WHERE l.batch_id = '{batch_id}'

      AND NOT EXISTS (
          SELECT 1
          FROM workspace.bronze.orders b
          WHERE b.batch_id = l.batch_id
            AND b.event_id = l.event_id
      )
    """)

    print("Bronze load completed.")


    # --------------------------------------------------
    # 3. COUNT INSERTED ROWS THIS RUN
    # --------------------------------------------------

    bronze_after = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM workspace.bronze.orders
        WHERE batch_id = '{batch_id}'
    """).first()["cnt"]

    bronze_rows = bronze_after - bronze_before

    print(f"Bronze inserted this run: {bronze_rows}")


    # --------------------------------------------------
    # 4. UPDATE AUDIT
    # --------------------------------------------------

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET bronze_rows = {bronze_rows}

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)

    print("Audit updated successfully.")


    # --------------------------------------------------
    # 5. SUMMARY
    # --------------------------------------------------

    print("----------------------------------")
    print("BRONZE PROCESSING COMPLETE")
    print("----------------------------------")
    print(f"Run ID:                  {run_id}")
    print(f"Batch ID:                {batch_id}")
    print(f"Bronze inserted this run:{bronze_rows}")
    print("----------------------------------")


except Exception as e:

    error_message = str(e).replace("'", "''")[:4000]

    spark.sql(f"""
        UPDATE workspace.control.etl_run_log

        SET
            end_timestamp = CURRENT_TIMESTAMP(),
            status = 'FAILED',
            error_message = '{error_message}'

        WHERE run_id = '{run_id}'
          AND status = 'RUNNING'
    """)

    print("Bronze processing FAILED.")
    print(error_message)

    raise

Run ID:   596f9a73-298f-478d-94d2-c240a58bf207
Batch ID: batch_006
Bronze load completed.
Bronze inserted this run: 407
Audit updated successfully.
----------------------------------
BRONZE PROCESSING COMPLETE
----------------------------------
Run ID:                  596f9a73-298f-478d-94d2-c240a58bf207
Batch ID:                batch_006
Bronze inserted this run:407
----------------------------------


In [0]:
%sql
SELECT
    run_id,
    batch_id,
    start_timestamp,
    end_timestamp,
    bronze_rows,
    status,
    error_message
FROM workspace.control.etl_run_log
ORDER BY start_timestamp DESC;

run_id,batch_id,start_timestamp,end_timestamp,bronze_rows,status,error_message
0bd99448-c8ce-40d2-8812-e7ba4aa694b2,batch_003,2026-08-29T18:35:52.832Z,null,765,RUNNING,null
cf648f21-44ea-4622-a6a9-26c9d7feed71,batch_002,2026-08-28T21:16:24.461Z,2026-08-28T21:17:24.481Z,null,FAILED,"[TABLE_OR_VIEW_NOT_FOUND] The table or view `workspace`.`bronze`.`orders_broken` cannot be found. Verify the spelling and correctness of the schema and catalog. Search path: [`system`.`session`, `system`.`builtin`, `system`.`ai`, `workspace`.`default`]. If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog. To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 2 pos 16; InsertIntoStatement UnresolvedRelation [workspace, bronze, orders_broken], [__required_write_privileges__=INSERT], false, false, false, false, false +- Project [order_id#27336, customer_id#27337, amount#27338, status#27339, order_date#27340, last_updated#27341, event_id#27342, batch_id#27343, source_file#27344, load_timestamp#27345] +- Filter ((batch_id#27343 = batch_002) AND NOT exists#27315 [batch_id#27343 && event_id#27342]) : +- Project [1 AS 1#27356] : +- Filter ((batch_id#27353 = outer(batch_id#27343)) AND (event_id#27352 = outer(event_id#27342))) : +- SubqueryAlias b : +- SubqueryAlias workspace.bronze.orders : +- Relation workspace.bronze.orders[order_id#27346,customer_id#27347,amount#27348,status#27349,order_date#27350,last_updated#27351,event_id#27352,batch_id#27353,source_file#27354,load_timestamp#27355] parquet +- SubqueryAlias l +- SubqueryAlias workspace.landing.orders +- Relation workspace.landing.orders[order_id#27336,customer_id#27337,amount#27338,status#27339,order_date#27340,last_updated#27341,event_id#27342,batch_id#27343,source_file#27344,load_timestamp#27345] parquet JVM stacktrace: org.apache.spark.sql.catalyst.ExtendedAnalysisException at org.apache.spark.sql.errors.QueryCompilationErrors$.tableOrViewNotFoundWithSearchPath(QueryCompilationErrors.scala:1612) at org.apache.spark.sql.catalyst.analysis.package$AnalysisErrorAt.tableNotFound(package.scala:97) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$1(CheckAnalysis.scala:411) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis0$1$adapted(CheckAnalysis.scala:407) at org.apache.spark.sql.catalyst.trees.TreeNode.foreach(TreeNode.scala:372) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0(CheckAnalysis.scala:407) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis0$(CheckAnalysis.scala:403) at org.apache.spark.sql.catalyst.analysis.Analyzer.checkAnalysis0(Analyzer.scala:717) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.$anonfun$checkAnalysis$1(CheckAnalysis.scala:388) at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18) at com.databricks.spark.util.FrameProfiler$.$anonfun$record$1(FrameProfiler.scala:114) at com.databricks.spark.util.FrameProfilerExporter$.maybeExportFrameProfiler(FrameProfilerExporter.scala:201) at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:105) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis(CheckAnalysis.scala:375) at org.apache.spark.sql.catalyst.analysis.CheckAnalysis.checkAnalysis$(CheckAnalysis.scala:371) at org.apache.spark.sql.catalyst.analysis.Analyzer.checkAnalysis(Analyzer.scala:717) at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.$anonfun$resolveInFixedPoint$1(HybridAnalyzer.scala:417) at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18) at org.apache.spark.sql.catalyst.QueryPlanningTracker$.withTracker(QueryPlanningTracker.scala:279) at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.resolveInFixedPoint(HybridAnalyzer.scala:417) at org.apache.spark.sql.catalyst.analysis.resolver.HybridAnalyzer.$anonfun$apply$1(HybridAnalyze

In [0]:
%sql
select * from control.etl_run_log
order by start_timestamp desc
limit 1

run_id,batch_id,pipeline_name,start_timestamp,end_timestamp,landing_rows,bronze_rows,silver_rows,gold_inserted,gold_updated,rejected_rows,status,error_message
e9f04c39-297c-465c-bf2b-d6c261a6c086,batch_003,orders_pipeline,2026-08-31T16:44:56.487Z,2026-08-31T16:57:55.340Z,1,766,765,1,0,7,SUCCESS,null
